In [11]:
from Bio import SeqIO
import numpy as np
import pandas as pd
from collections import Counter


In [12]:
#Я взял мультифасту гена NF1 и все объединил (мб там какие-то варианты сплайсинга я хз), сам файл прикладывать не буду
records = SeqIO.parse("gene.fasta", "fasta")
seq = ''.join(str(record.seq).upper() for record in records)
seq = ''.join(base for base in seq if base in "ACGT")

alphabet = ['A', 'C', 'G', 'T']
idx = {b: i for i, b in enumerate(alphabet)}

In [13]:
# 2. Подсчет частот 16 динуклеотидов
dinucs = [seq[i:i+2] for i in range(len(seq) - 1)]
dinuc_counts = Counter(dinucs)

# Убедимся, что все 16 присутствуют в словаре
all_dinucs = [a + b for a in alphabet for b in alphabet]
for d in all_dinucs:
    dinuc_counts[d] += 0

print("Dinucleotide counts:")
for d in all_dinucs:
    print(f"{d}: {dinuc_counts[d]}")

Dinucleotide counts:
AA: 81545
AC: 38022
AG: 55155
AT: 70692
CA: 53043
CC: 36051
CG: 6905
CT: 58281
GA: 45861
GC: 31508
GG: 37444
GT: 48932
TA: 64965
TC: 48699
TG: 64241
TT: 106069


In [14]:
# 3. Матрица переходов 4x4
count_matrix = np.zeros((4, 4), dtype=int)

for d, c in dinuc_counts.items():
    if len(d) == 2 and d[0] in idx and d[1] in idx:
        i = idx[d[0]]
        j = idx[d[1]]
        count_matrix[i, j] = c

row_sums = count_matrix.sum(axis=1, keepdims=True)

# Чтобы избежать деления на ноль
P = np.divide(count_matrix, row_sums, where=(row_sums != 0))
P[row_sums.flatten() == 0] = 0

print("\nTransition matrix P:")
print(pd.DataFrame(P, index=alphabet, columns=alphabet))


Transition matrix P:
          A         C         G         T
A  0.332275  0.154930  0.224743  0.288052
C  0.343810  0.233673  0.044756  0.377761
G  0.280076  0.192421  0.228673  0.298830
T  0.228771  0.171491  0.226221  0.373517


In [ ]:
# 4. Проверка сумм строк
print("\nRow sums:")
print(P.sum(axis=1))

In [ ]:
# 5. Стационарное распределение: pi = pi P
# Решаем через собственный вектор для собственного значения 1 матрицы P^T
eigvals, eigvecs = np.linalg.eig(P.T)
k = np.argmin(np.abs(eigvals - 1))
pi = np.real(eigvecs[:, k])
pi = pi / pi.sum()

# Чистим возможные численные артефакты
pi = np.real_if_close(pi)
pi = pi / pi.sum()

print("\nStationary distribution pi:")
for b, val in zip(alphabet, pi):
    print(f"{b}: {val:.6f}")


In [ ]:
# 6. Наблюдаемые частоты отдельных нуклеотидов
mono_counts = Counter(seq)
mono_total = sum(mono_counts[b] for b in alphabet)
obs_freq = np.array([mono_counts[b] / mono_total for b in alphabet])

print("\nObserved mono-nucleotide frequencies:")
for b, val in zip(alphabet, obs_freq):
    print(f"{b}: {val:.6f}")


In [ ]:
# 7. Сравнение
diff = pi - obs_freq
print("\nDifference (pi - observed):")
for b, val in zip(alphabet, diff):
    print(f"{b}: {val:.6f}")


In [8]:
# 8. Красивый вывод в таблицах
dinuc_df = pd.DataFrame(
    [{"dinucleotide": d, "count": dinuc_counts[d]} for d in all_dinucs]
)

P_df = pd.DataFrame(P, index=alphabet, columns=alphabet)
compare_df = pd.DataFrame({
    "base": alphabet,
    "stationary_pi": pi,
    "observed_freq": obs_freq,
    "difference": diff
})

print("\nDinucleotide table:")
print(dinuc_df)

print("\nComparison table:")
print(compare_df)



Dinucleotide table:
   dinucleotide   count
0            AA   81545
1            AC   38022
2            AG   55155
3            AT   70692
4            CA   53043
5            CC   36051
6            CG    6905
7            CT   58281
8            GA   45861
9            GC   31508
10           GG   37444
11           GT   48932
12           TA   64965
13           TC   48699
14           TG   64241
15           TT  106069

Comparison table:
  base  stationary_pi  observed_freq    difference
0    A       0.289604       0.289605 -8.383107e-07
1    C       0.182060       0.182060  2.148418e-07
2    G       0.193229       0.193229  2.280223e-07
3    T       0.335107       0.335107  3.954466e-07
